# Isolation Experiment Acoustic Analysis Pipeline
<hr>
<i> dr n. med. inż. Maksymilian A. Brzezicki, BSc MB ChB DPhil <br> Department of Neural Engineering and Space Medicine, University of Zielona Góra / Clinical Fellow, Jesus College, University of Oxford  </i><br>
Analysis code accompanying: Reduced vocal variability across a short-term analog space mission 
<hr>
<br>

---
## 1. Setup and Configuration

In [ ]:
# ============================================================
# CONFIGURATION - MODIFY THESE PARAMETERS
# ============================================================

DATA_FILE = "data/results_20260101_223957.csv"

# Correlation threshold for feature filtering
CORRELATION_THRESHOLD = 0.8

# Significance level
ALPHA = 0.05


# Random seed for reproducibility
RANDOM_SEED = 42

# ============================================================

In [ ]:
# ============================================================
# IMPORTS - All dependencies consolidated
# ============================================================

import sys
import time
import re
import warnings
from pathlib import Path
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, linregress, friedmanchisquare, wilcoxon, ttest_rel

# Optional dependencies with auto-install
try:
    from tqdm import tqdm
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'tqdm', '--break-system-packages', '-q'])
    from tqdm import tqdm

# Statsmodels
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from statsmodels.stats.multitest import multipletests
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print("Warning: statsmodels not available. Some analyses will use fallback methods.")

# Scikit-learn (with proper fallback - the previous fallback had a typo
# that would crash if sklearn were truly missing)
try:
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
    from sklearn.svm import SVR
    from sklearn.decomposition import PCA
    from sklearn.model_selection import cross_val_score, KFold, cross_val_predict, LeaveOneGroupOut
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
    from sklearn.base import clone
    HAS_SKLEARN = True
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'scikit-learn', '--break-system-packages', '-q'])
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
    from sklearn.svm import SVR
    from sklearn.decomposition import PCA
    from sklearn.model_selection import cross_val_score, KFold, cross_val_predict, LeaveOneGroupOut
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
    from sklearn.base import clone
    HAS_SKLEARN = True

# XGBoost (optional)
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("Warning: XGBoost not available.")

# MERF (optional)
try:
    from merf import MERF
    HAS_MERF = True
except ImportError:
    HAS_MERF = False
    print("Warning: MERF not available. Install with: pip install merf")

# Targeted warning suppression only (3h):
# We DO want to see convergence warnings from MixedLM and MERF.
# Suppress only known-noisy categories.
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*p-value.*')

np.random.seed(RANDOM_SEED)

print("Imports complete.")
print(f"  - statsmodels: {chr(10003) if HAS_STATSMODELS else chr(10007)}")
print(f"  - scikit-learn: {chr(10003) if HAS_SKLEARN else chr(10007)}")
print(f"  - XGBoost: {chr(10003) if HAS_XGBOOST else chr(10007)}")
print(f"  - MERF: {chr(10003) if HAS_MERF else chr(10007)}")


In [ ]:
# Targeted warning suppression
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='statsmodels')
warnings.filterwarnings('ignore', message='.*p-value.*')

In [ ]:
# ============================================================
# CONFIGURATION CLASS & VISUALIZATION SETTINGS
# ============================================================

@dataclass
class Config:
    data_path: str = DATA_FILE
    filename_col: str = "filename"
    subject_col: str = "subject_ID"
    exclude_cols: Tuple[str, ...] = ("filename", "filename_AsWritten", "subject_ID", "file", "mel_plot_path",
                                      "mel_spec_mean", "mel_spec_std", "mel_spec_max", "mel_spec_min")
    alpha: float = ALPHA
    correlation_threshold: float = CORRELATION_THRESHOLD
    random_seed: int = RANDOM_SEED
    primary_color: str = "#1f77b4"
    accent_color: str = "#0d4f8b"
    output_dir: str = "analysis_outputs"

config = Config()
Path(config.output_dir).mkdir(exist_ok=True)

# Poster-ready font sizes
FONT = {'title': 14, 'axes': 12, 'tick': 11, 'legend': 9, 'annotation': 11, 'suptitle': 16}

# Colors for visualizations
PRIMARY_COLOR = config.primary_color
NEGATIVE_COLOR = '#d62728'

def setup_style():
    plt.rcParams.update({
        'figure.facecolor': 'white', 'axes.facecolor': 'white',
        'axes.edgecolor': '#333333', 'axes.linewidth': 1.2, 'axes.grid': False,
        'font.size': FONT['tick'], 'axes.titlesize': FONT['title'],
        'axes.labelsize': FONT['axes'], 'xtick.labelsize': FONT['tick'],
        'ytick.labelsize': FONT['tick'], 'legend.fontsize': FONT['legend'],
        'figure.dpi': 100,
    })

setup_style()

In [ ]:
# ============================================================
# FEATURE NAME MAPPING (Consolidated)
# ============================================================

FEATURE_NAMES = {
    # MFCC features
    'mfcc1_sma3_amean': ('MFCC 1 (Mean)', 'coefficient'),
    'mfcc2_sma3_amean': ('MFCC 2 (Mean)', 'coefficient'),
    'mfcc3_sma3_amean': ('MFCC 3 (Mean)', 'coefficient'),
    'mfcc4_sma3_amean': ('MFCC 4 (Mean)', 'coefficient'),
    'mfcc1V_sma3nz_amean': ('MFCC 1 Voiced (Mean)', 'coefficient'),
    'mfcc2V_sma3nz_amean': ('MFCC 2 Voiced (Mean)', 'coefficient'),
    'mfcc3V_sma3nz_amean': ('MFCC 3 Voiced (Mean)', 'coefficient'),
    'mfcc4V_sma3nz_amean': ('MFCC 4 Voiced (Mean)', 'coefficient'),
    'mfcc1_sma3_stddevNorm': ('MFCC 1 (Variability)', 'norm. SD'),
    'mfcc2_sma3_stddevNorm': ('MFCC 2 (Variability)', 'norm. SD'),
    'mfcc3_sma3_stddevNorm': ('MFCC 3 (Variability)', 'norm. SD'),
    'mfcc4_sma3_stddevNorm': ('MFCC 4 (Variability)', 'norm. SD'),
    'mfcc1V_sma3nz_stddevNorm': ('MFCC 1 Voiced (Variability)', 'norm. SD'),
    'mfcc2V_sma3nz_stddevNorm': ('MFCC 2 Voiced (Variability)', 'norm. SD'),
    'mfcc3V_sma3nz_stddevNorm': ('MFCC 3 Voiced (Variability)', 'norm. SD'),
    'mfcc4V_sma3nz_stddevNorm': ('MFCC 4 Voiced (Variability)', 'norm. SD'),
    # Pitch features
    'F0semitoneFrom27.5Hz_sma3nz_amean': ('Pitch (Mean)', 'semitones'),
    'F0semitoneFrom27.5Hz_sma3nz_stddevNorm': ('Pitch (Variability)', 'norm. SD'),
    'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2': ('Pitch Range', 'semitones'),
    'F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope': ('F0 Mean Falling Slope', 'semitones/s'),
    # Formant features
    'F1frequency_sma3nz_amean': ('Formant F1 (Mean)', 'Hz'),
    'F1frequency_sma3nz_stddevNorm': ('Formant F1 (Variability)', 'norm. SD'),
    'F2frequency_sma3nz_amean': ('Formant F2 (Mean)', 'Hz'),
    'F2frequency_sma3nz_stddevNorm': ('Formant F2 (Variability)', 'norm. SD'),
    'F3frequency_sma3nz_amean': ('Formant F3 (Mean)', 'Hz'),
    'F3frequency_sma3nz_stddevNorm': ('Formant F3 (Variability)', 'norm. SD'),
    'F1bandwidth_sma3nz_amean': ('F1 Bandwidth (Mean)', 'Hz'),
    'F2bandwidth_sma3nz_amean': ('F2 Bandwidth (Mean)', 'Hz'),
    'F3bandwidth_sma3nz_amean': ('F3 Bandwidth (Mean)', 'Hz'),
    'F1amplitudeLogRelF0_sma3nz_amean': ('F1 Amplitude rel. F0 (Mean)', 'dB'),
    'F1amplitudeLogRelF0_sma3nz_stddevNorm': ('F1 Amp. rel. F0 (Variability)', 'norm. SD'),
    'F2amplitudeLogRelF0_sma3nz_amean': ('F2 Amplitude rel. F0 (Mean)', 'dB'),
    'F2amplitudeLogRelF0_sma3nz_stddevNorm': ('F2 Amp. rel. F0 (Variability)', 'norm. SD'),
    'F3amplitudeLogRelF0_sma3nz_amean': ('F3 Amplitude rel. F0 (Mean)', 'dB'),
    'F3amplitudeLogRelF0_sma3nz_stddevNorm': ('F3 Amp. rel. F0 (Variability)', 'norm. SD'),
    # Loudness features
    'loudness_sma3_amean': ('Loudness (Mean)', 'sone'),
    'loudness_sma3_stddevNorm': ('Loudness (Variability)', 'norm. SD'),
    'loudness_sma3_pctlrange0-2': ('Loudness Range', 'sone'),
    # Voice quality features
    'HNRdBACF_sma3nz_amean': ('HNR (Mean)', 'dB'),
    'HNRdBACF_sma3nz_stddevNorm': ('HNR (Variability)', 'norm. SD'),
    'jitterLocal_sma3nz_amean': ('Jitter (Mean)', '%'),
    'jitterLocal_sma3nz_stddevNorm': ('Jitter (Variability)', 'norm. SD'),
    'shimmerLocaldB_sma3nz_amean': ('Shimmer (Mean)', 'dB'),
    'shimmerLocaldB_sma3nz_stddevNorm': ('Shimmer (Variability)', 'norm. SD'),
    'logRelF0-H1-H2_sma3nz_amean': ('H1-H2 (Mean)', 'dB'),
    'logRelF0-H1-H2_sma3nz_stddevNorm': ('H1-H2 (Variability)', 'norm. SD'),
    # Segmentation features
    'VoicedSegmentsPerSec': ('Voiced Segments/Sec', 'segments/s'),
    'MeanVoicedSegmentLengthSec': ('Mean Voiced Seg Length', 's'),
}

def get_feature_name(feature, max_len=None):
    """Convert raw feature name to display name and unit."""
    if feature in FEATURE_NAMES:
        name, unit = FEATURE_NAMES[feature]
        if max_len and len(name) > max_len:
            name = name[:max_len-2] + '..'
        return (name, unit)
    
    # Fallback: clean up feature name
    name = feature.replace('_sma3nz', '').replace('_sma3', '') # type: ignore
    name = name.replace('_amean', ' (Mean)').replace('_stddevNorm', ' (Var.)') # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]
    name = name.replace('_pctlrange0-2', ' Range').replace('_', ' ') # type: ignore
    name = re.sub(r'([a-z])([A-Z])', r'\1 \2', name)
    name = ' '.join(name.split())
    
    if max_len and len(name) > max_len:
        name = name[:max_len-2] + '..'
    
    return (name[:35] if not max_len else name, 'a.u.')

In [ ]:
# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def remove_gridlines(ax):
    """Remove gridlines and top/right spines from axes."""
    ax.grid(False)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

def get_blue_palette(n):
    """Generate a blue color palette."""
    
    return sns.color_palette('Blues', n_colors=n).as_hex()

def fdr_correction(p_values, alpha=0.05):
    """Benjamini-Hochberg FDR correction."""
    p_values = np.asarray(p_values)
    n = len(p_values)
    if n == 0:
        return np.array([]), np.array([])
    sorted_idx = np.argsort(p_values)
    sorted_p = p_values[sorted_idx]
    adjusted = np.zeros(n)
    for i in range(n-1, -1, -1):
        if i == n-1:
            adjusted[sorted_idx[i]] = sorted_p[i]
        else:
            adjusted[sorted_idx[i]] = min(adjusted[sorted_idx[i+1]], sorted_p[i] * n / (i + 1))
    return adjusted < alpha, np.minimum(adjusted, 1.0)

class ProgressBar:
    """Simple progress bar for training/testing."""
    
    def __init__(self, total, prefix='Progress', length=40):
        self.total = total
        self.prefix = prefix
        self.length = length
        self.current = 0
        self.start_time = time.time()
    
    def update(self, step=1, suffix=''):
        self.current += step
        percent = self.current / self.total
        filled = int(self.length * percent)
        bar = '█' * filled + '░' * (self.length - filled)
        elapsed = time.time() - self.start_time
        
        if self.current > 0:
            eta = (elapsed / self.current) * (self.total - self.current)
            time_str = f'ETA: {eta:.1f}s' if self.current < self.total else f'Done in {elapsed:.1f}s'
        else:
            time_str = ''
        
        sys.stdout.write(f'\r  {self.prefix} |{bar}| {self.current}/{self.total} ({percent:.0%}) {suffix} {time_str}')
        sys.stdout.flush()
        
        if self.current >= self.total:
            print()  # New line when complete
    
    def reset(self, prefix=None):
        self.current = 0
        self.start_time = time.time()
        if prefix:
            self.prefix = prefix

---
## 1. Data Loading

In [ ]:
# ============================================================
# DATA LOADING FUNCTIONS
# ============================================================

def extract_date(filename):
    """Extract day and month from filename (format: DD.MM... or D.MM...)."""
    match = re.match(r'^(\d{1,2})\.(\d{2})', str(filename))
    return (int(match.group(1)), int(match.group(2))) if match else None

def compute_days(df, subject_col, filename_col, anchor_year=None):
    """
    Compute relative day numbers from filenames for each subject.

    Robust to year boundaries: if recordings within a subject cross a
    year boundary (e.g. December into January), months 1-3 occurring AFTER
    any month >= 10 within that subject are bumped to anchor_year + 1.

    If anchor_year is None, uses 2026 as default (matches v2 behaviour
    for the existing dataset).
    """
    df = df.copy()
    if anchor_year is None:
        anchor_year = 2026

    date_info = df[filename_col].apply(extract_date)
    df['_day'] = date_info.apply(lambda x: x[0] if x else None)
    df['_month'] = date_info.apply(lambda x: x[1] if x else None)

    # Default year for everyone
    df['_year'] = anchor_year

    # For each subject, identify whether they span a year boundary,
    # and bump early months (1-3) that occur after any late month (>=10).
    for subj, idx in df.groupby(subject_col).groups.items():
        sub = df.loc[idx]
        months = sub['_month'].values
        # Need at least one late and one early month to cross a boundary
        if pd.notna(months).any() and (months >= 10).any() and (months <= 3).any():
            # Walk through the rows in their original order; once we see
            # a late month, any subsequent early month gets bumped.
            late_seen = False
            for ridx, m in zip(sub.index, months):
                if pd.notna(m) and m >= 10:
                    late_seen = True
                if late_seen and pd.notna(m) and m <= 3:
                    df.at[ridx, '_year'] = anchor_year + 1

    df['_date'] = pd.to_datetime(
        df['_day'].astype('Int64').astype(str) + '-' +
        df['_month'].astype('Int64').astype(str) + '-' +
        df['_year'].astype(str),
        format='%d-%m-%Y', errors='coerce'
    )
    df = df.sort_values([subject_col, '_date']).reset_index(drop=True)
    first_dates = df.groupby(subject_col)['_date'].transform('min')
    df['day'] = (df['_date'] - first_dates).dt.days + 1
    return df.drop(columns=['_day', '_month', '_year', '_date'])

# Load and preprocess data
df_raw = pd.read_csv(config.data_path)
df = compute_days(df_raw, config.subject_col, config.filename_col)

# Identify feature columns
# Build candidate feature list, then auto-drop any non-numeric columns
# (defensive: catches string columns the user may not have known to exclude)
candidate_cols = [c for c in df.columns if c not in config.exclude_cols and c != 'day']
numeric_mask = df[candidate_cols].apply(pd.to_numeric, errors='coerce').notna().any()
non_numeric = [c for c in candidate_cols if not numeric_mask.get(c, False) or df[c].dtype == 'object']
# Use only columns that are already numeric dtype
feature_cols_initial = [c for c in candidate_cols if pd.api.types.is_numeric_dtype(df[c])]
auto_dropped = [c for c in candidate_cols if c not in feature_cols_initial]
if auto_dropped:
    print(f"[INFO] Auto-dropped {len(auto_dropped)} non-numeric column(s) from features: {auto_dropped}")
n_features_initial = len(feature_cols_initial)
n_subjects = df[config.subject_col].nunique()
n_observations = len(df)

print(f"Data loaded: {n_observations} recordings, {n_subjects} subjects")
print(f"Observations per subject: {n_observations/n_subjects:.1f} (mean)")
print(f"Initial features: {n_features_initial}")
print(f"Day range: {df['day'].min()} - {df['day'].max()}")
print(f"\n[NOTE] SAMPLE SIZE: n={n_subjects} subjects. Results are exploratory.")


---
## 2. Feature Selection: Correlation Filter, Audit, Manual Rescue

In [ ]:
# ============================================================
# CANONICAL CORRELATION FILTER (single source of truth)
# ============================================================
# This function is the ONLY correlation filter used in the entire notebook.
# It is called from:
#   - Here (top-level filter)
#   - The audit cell (immediately below)
#   - The ML pipeline cell (inside each CV fold)
#   - The PDF report cell (for feature counts)
# All call sites give identical numbers by construction.

def remove_correlated_canonical(df, features, threshold):
    """
    Greedy correlation filter with early-exit.

    Walks features in column order. For each feature i still alive,
    checks every later feature j still alive and drops j if
    |corr(i, j)| > threshold.

    Key property: a feature that has already been removed cannot
    trigger further removals. This makes every removal traceable to
    exactly one surviving "kept" partner, and gives reproducible counts
    across all call sites.

    Parameters
    ----------
    df : DataFrame containing the features.
    features : list of column names (order matters).
    threshold : absolute Pearson correlation cutoff (e.g. 0.85).

    Returns
    -------
    kept : list of feature names that survived the filter, in original order.
    """
    if threshold >= 1.0:
        return list(features)

    abs_corr = df[features].corr().abs()
    to_remove = set()

    for i in range(len(features)):
        if features[i] in to_remove:
            continue  # already-removed features cannot trigger further removals
        for j in range(i + 1, len(features)):
            if features[j] in to_remove:
                continue
            if abs_corr.iloc[i, j] > threshold:
                to_remove.add(features[j])

    return [f for f in features if f not in to_remove]


# Apply the filter at top level
feature_cols = remove_correlated_canonical(
    df, feature_cols_initial, config.correlation_threshold
)
n_features_after_corr = len(feature_cols)
n_removed_corr = n_features_initial - n_features_after_corr

print(f"\nCorrelation threshold: |r| > {config.correlation_threshold}")
print(f"Features removed by correlation filter: {n_removed_corr}")
print(f"Features remaining: {n_features_after_corr}")


In [ ]:
# ============================================================
# CORRELATION FILTERING AUDIT
# ============================================================
# Show which features were removed, which were kept, and the specific
# correlated partner that triggered each removal. Uses the same canonical
# filter as the rest of the notebook, so counts ALWAYS match.

def audit_correlation_filtering(df, features, threshold):
    """
    Replay the canonical greedy correlation filter and record every
    removal decision. Identical logic to remove_correlated_canonical(),
    but also returns a DataFrame describing each (removed, kept, r) pair.
    """
    corr = df[features].corr()      # signed, for reporting
    abs_corr = corr.abs()
    to_remove = set()
    decisions = []

    for i in range(len(features)):
        if features[i] in to_remove:
            continue
        for j in range(i + 1, len(features)):
            if features[j] in to_remove:
                continue
            r_abs = abs_corr.iloc[i, j]
            if r_abs > threshold:
                to_remove.add(features[j])
                decisions.append({
                    'removed_feature': features[j],
                    'kept_feature': features[i],
                    'r_value': float(corr.iloc[i, j]),
                    'abs_r': float(r_abs),
                })

    kept = [f for f in features if f not in to_remove]
    return pd.DataFrame(decisions), to_remove, kept


audit_df, removed_set, kept_list = audit_correlation_filtering(
    df, feature_cols_initial, config.correlation_threshold
)

# CONSISTENCY CHECK - audit and canonical filter must agree exactly
assert kept_list == feature_cols, (
    f"Audit/main filter mismatch! Audit kept {len(kept_list)}, "
    f"main kept {len(feature_cols)}. This should be impossible."
)
print(f"[OK] Audit and main filter agree exactly: {len(kept_list)} features retained.\n")

audit_df = audit_df.sort_values('abs_r', ascending=False).reset_index(drop=True)
audit_df['removed_name'] = audit_df['removed_feature'].apply(lambda x: get_feature_name(x)[0])
audit_df['kept_name'] = audit_df['kept_feature'].apply(lambda x: get_feature_name(x)[0])

print("=" * 90)
print(f"CORRELATION FILTERING AUDIT (threshold |r| > {config.correlation_threshold})")
print("=" * 90)
print(f"\nInitial features:  {len(feature_cols_initial)}")
print(f"Features removed:  {len(removed_set)}")
print(f"Features retained: {len(kept_list)}")
print(f"Removal decisions: {len(audit_df)} (one per dropped feature)")
print()
print(f"{'Removed feature':<35} {'paired with kept feature':<35} {'r':>8}")
print("-" * 90)
for _, row in audit_df.iterrows():
    print(f"{row['removed_name'][:33]:<35} {row['kept_name'][:33]:<35} {row['r_value']:>+8.3f}")

print()
print("=" * 90)
print("FEATURES RETAINED AFTER FILTERING")
print("=" * 90)
for f in kept_list:
    name, unit = get_feature_name(f)
    print(f"  {name:<35} ({unit})")

audit_df[['removed_feature', 'kept_feature', 'r_value', 'abs_r',
          'removed_name', 'kept_name']].to_csv(
    f"{config.output_dir}/correlation_filter_audit.csv", index=False
)
print(f"\n[OK] Audit table saved to {config.output_dir}/correlation_filter_audit.csv")


In [ ]:
# ============================================================
# MANUAL FEATURE RESCUE (post-audit)
# ============================================================
# Add features back into feature_cols that the canonical filter removed.
# Runs AFTER the audit, so the audit reflects the pure filter behaviour
# and this cell records any deviations from it.

RESCUED_FEATURES = [
    'HNRdBACF_sma3nz_amean',
    'MeanUnvoicedSegmentLength',
]

print("=" * 70)
print("MANUAL FEATURE RESCUE (post-audit)")
print("=" * 70)

n_before = len(feature_cols)
rescued_added = []

for feat in RESCUED_FEATURES:
    if feat not in feature_cols_initial:
        print(f"[SKIP]    {feat}: not in initial feature list (check column name)")
        continue

    if feat in feature_cols:
        print(f"[ALREADY] {feat}: already retained by filter, no rescue needed")
        continue

    # Insert at the position it had in the original column order
    original_idx = feature_cols_initial.index(feat)
    insert_at = len(feature_cols)
    for i, kept in enumerate(feature_cols):
        if feature_cols_initial.index(kept) > original_idx:
            insert_at = i
            break

    feature_cols.insert(insert_at, feat)
    rescued_added.append(feat)
    print(f"[RESCUED] {feat}")

# Update count variables that downstream cells (15, 33, etc.) use
n_features_after_corr = len(feature_cols)
n_removed_corr = n_features_initial - n_features_after_corr

print("-" * 70)
print(f"Filter-retained features:    {n_before}")
print(f"Manually rescued:            {len(rescued_added)}")
print(f"Total features for analysis: {n_features_after_corr}")
print(f"Total removed:               {n_removed_corr}")

---
## 3. Trend Analysis (Random-Intercept Mixed-Effects Models)

In [ ]:
# ============================================================
# TREND ANALYSIS FUNCTIONS
# ============================================================

def analyze_individual_trends(df, features, subject_col):
    """
    Analyze trends for each subject individually.

    Captures slope, slope SE, 95% CI, R^2 and p-value per subject per feature.
    These per-subject p-values are LOW POWER (typically ~14 timepoints per
    subject), so should not be the primary cross-subject claim.
    """
    results = []
    for subject in df[subject_col].unique():
        subj_df = df[df[subject_col] == subject]
        days = subj_df['day'].values
        for feature in features:
            values = subj_df[feature].values
            mask = ~(np.isnan(days) | np.isnan(values))
            n_valid = mask.sum()
            if n_valid >= 3:
                slope, intercept, r, p, se = linregress(days[mask], values[mask])
                # 95% CI for slope using t-distribution
                if n_valid > 2 and not np.isnan(se):
                    t_crit = stats.t.ppf(0.975, df=n_valid - 2)
                    ci_low = slope - t_crit * se
                    ci_high = slope + t_crit * se
                else:
                    ci_low, ci_high = np.nan, np.nan
                results.append({
                    'subject': subject, 'feature': feature, 'n': n_valid,
                    'slope': slope, 'slope_se': se,
                    'slope_ci_low': ci_low, 'slope_ci_high': ci_high,
                    'r_value': r, 'r_squared': r**2, 'p_value': p,
                    'significant': p < config.alpha,
                    'direction': 'increasing' if slope > 0 else 'decreasing'
                })
    return pd.DataFrame(results)

def analyze_group_trends(df, features, subject_col):
    """
    Analyze trends at group level using mixed-effects models with random
    intercepts per subject. Returns slope, slope SE, 95% CI, p, and a
    convergence flag.
    """
    results = []
    for feature in features:
        data = df[[subject_col, 'day', feature]].dropna()
        converged = True
        method_used = 'mixed'
        if len(data) >= 10:
            if HAS_STATSMODELS:
                try:
                    model = sm.MixedLM.from_formula(
                        f'Q("{feature}") ~ day',
                        groups=data[subject_col], data=data
                    )
                    result = model.fit(disp=False)
                    if not result.converged:
                        converged = False
                    slope = result.fe_params['day']
                    p = result.pvalues['day']
                    se = result.bse_fe['day'] if hasattr(result, 'bse_fe') else result.bse['day']
                    # 95% CI using normal approximation (mixed model)
                    ci_low = slope - 1.96 * se
                    ci_high = slope + 1.96 * se
                    r = np.corrcoef(data[feature], result.fittedvalues)[0, 1]
                except Exception:
                    # Fallback to simple linear regression if mixed model fails
                    slope, _, r, p, se = linregress(data['day'].values, data[feature].values)
                    ci_low = slope - 1.96 * se
                    ci_high = slope + 1.96 * se
                    method_used = 'ols_fallback'
                    converged = False
            else:
                slope, _, r, p, se = linregress(data['day'].values, data[feature].values)
                ci_low = slope - 1.96 * se
                ci_high = slope + 1.96 * se
                method_used = 'ols_fallback'
            results.append({
                'feature': feature, 'slope': slope, 'slope_se': se,
                'slope_ci_low': ci_low, 'slope_ci_high': ci_high,
                'r_value': r, 'r_squared': r**2,
                'p_value': p,
                'direction': 'increasing' if slope > 0 else 'decreasing',
                'converged': converged, 'method': method_used
            })
    return pd.DataFrame(results)


In [ ]:
# Run trend analyses
individual_trends = analyze_individual_trends(df, feature_cols, config.subject_col)
group_trends = analyze_group_trends(df, feature_cols, config.subject_col)

# Report convergence diagnostics for mixed models
if 'converged' in group_trends.columns:
    n_failed = (~group_trends['converged']).sum()
    if n_failed > 0:
        print(f"[WARN] {n_failed}/{len(group_trends)} mixed models did not converge "
              f"or fell back to OLS. See `group_trends['method']` for details.")

# FDR correction (Benjamini-Hochberg) for the group-level day-slope p-values.
# BH controls the expected proportion of false discoveries among the rejected
# hypotheses, which is the appropriate correction for an exploratory analysis
# across many features.
if len(group_trends) > 0:
    if HAS_STATSMODELS:
        _, p_corr_bh, _, _ = multipletests(group_trends['p_value'].values, method='fdr_bh')
    else:
        _, p_corr_bh = fdr_correction(group_trends['p_value'].values)
    group_trends['p_fdr'] = p_corr_bh
    group_trends['sig_raw'] = group_trends['p_value'] < config.alpha
    group_trends['sig_fdr'] = p_corr_bh < config.alpha
    group_trends = group_trends.sort_values('p_value').reset_index(drop=True)

# FDR correction for individual-subject trends (per-subject FDR over features)
individual_trends['p_fdr'] = np.nan
individual_trends['sig_fdr'] = False
for subj in individual_trends['subject'].unique():
    mask = individual_trends['subject'] == subj
    pvals = individual_trends.loc[mask, 'p_value'].values
    if len(pvals) > 0:
        if HAS_STATSMODELS:
            _, p_corr_ind, _, _ = multipletests(pvals, method='fdr_bh')
        else:
            _, p_corr_ind = fdr_correction(pvals)
        individual_trends.loc[mask, 'p_fdr'] = p_corr_ind
        individual_trends.loc[mask, 'sig_fdr'] = p_corr_ind < config.alpha

# Identify the FDR-significant features - this is the headline set
sig_group = group_trends[group_trends['sig_fdr']].copy().sort_values('r_squared', ascending=False)
print(f"\nGroup-level FDR-BH significant features: {len(sig_group)}")


In [ ]:
# ============================================================
# HEADLINE COUNTS
# ============================================================
n_group_sig_raw = group_trends['sig_raw'].sum() if len(group_trends) > 0 else 0
n_group_sig_fdr = group_trends['sig_fdr'].sum() if len(group_trends) > 0 else 0

print("=" * 70)
print("HEADLINE COUNTS")
print("=" * 70)
print(f"Initial features:                                  {n_features_initial:>5}")
print(f"After correlation filter (|r| > {config.correlation_threshold}, with rescues): {len(feature_cols):>5}")
print(f"Group-level significant (raw p < 0.05):            {n_group_sig_raw:>5}")
print(f"Group-level significant (FDR-BH p < 0.05):         {n_group_sig_fdr:>5}  <-- HEADLINE")
print()
print(f"The {n_group_sig_fdr} FDR-BH significant features are the headline set.")


---
## 4. Headline Figure: Within-Subject Day Effect for the 9 FDR-Significant Features

Subject means are subtracted from each subject's values, so between-subject baseline differences are removed and the population-level day effect becomes visible. Slopes, confidence intervals, and FDR-corrected p-values come from the random-intercept mixed-effects model fit on the original (uncentred) data.

In [ ]:
def plot_group_trend_style(df, feature, subject_col, r_sq, ax):
    max_day = int(df['day'].max())
    subjects = sorted(df[subject_col].unique())
    colors = get_blue_palette(len(subjects))
    
    for i, subj in enumerate(subjects):
        subj_df = df[df[subject_col] == subj]
        ax.scatter(subj_df['day'], subj_df[feature], color=colors[i], s=35, alpha=0.7,
                   edgecolors='white', linewidth=0.3, label=subj)
    
    daily_mean = df.groupby('day')[feature].mean()
    daily_std = df.groupby('day')[feature].std()
    ax.errorbar(daily_mean.index, daily_mean.values, yerr=daily_std.values, fmt='o-',
                color=config.accent_color, markersize=7, linewidth=2, capsize=3, zorder=10,
                label='Group Mean ± SD')
    
    clean = df[['day', feature]].dropna()
    if len(clean) >= 2:
        z = np.polyfit(clean['day'], clean[feature], 1)
        p = np.poly1d(z)
        ax.plot(np.linspace(1, max_day, 100), p(np.linspace(1, max_day, 100)),
                '--', color=config.primary_color, alpha=0.8, linewidth=2, label='Group Trend')
    
    remove_gridlines(ax)
    ax.set_xlabel('Day')
    _, unit = get_feature_name(feature)
    ax.set_ylabel(unit)
    ax.set_xticks(range(1, max_day + 1))
    ax.text(0.02, 0.98, f'R² = {r_sq:.3f}', transform=ax.transAxes, fontsize=FONT['annotation'], va='top')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=FONT['legend']-1, frameon=False)

In [ ]:
# ============================================================
# WITHIN-SUBJECT DAY EFFECT (SUBJECT-MEAN-CENTRED)
# ============================================================
# This visualisation removes between-subject baseline differences by
# subtracting each subject's own mean from their values. What remains
# is the within-subject deviation from baseline at each day, so the
# population-level day effect becomes visible without being obscured
# by the (much larger) between-subject variance.
#
# Annotation: each panel shows the population-level slope from the
# random-intercept mixed-effects model, with 95% CI and FDR-corrected
# p-value. These are in original (un-centred) units and come from the
# group_trends DataFrame, so they match the headline analysis.

def plot_group_trend_centred(df, feature, subject_col, slope, ci_low, ci_high, p_fdr, ax):
    """Plot subject-mean-centred values to show the within-subject day effect."""
    max_day = int(df['day'].max())
    min_day = int(df['day'].min())
    subjects = sorted(df[subject_col].unique())
    colors = get_blue_palette(len(subjects))

    # Compute subject-mean-centred values (subject means subtracted)
    df_plot = df[[subject_col, 'day', feature]].dropna().copy()
    subject_means = df_plot.groupby(subject_col)[feature].transform('mean')
    df_plot['centred'] = df_plot[feature] - subject_means

    # Per-subject scatter only (no connecting lines)
    for i, subj in enumerate(subjects):
        subj_df = df_plot[df_plot[subject_col] == subj]
        ax.scatter(subj_df['day'], subj_df['centred'],
                   color=colors[i], s=35, alpha=0.6,
                   edgecolors='white', linewidth=0.3, label=subj)

    # Reference line at zero (each subject's own baseline)
    ax.axhline(y=0, color='gray', linewidth=0.8, linestyle=':', alpha=0.6, zorder=1)

    # Group mean of centred values per day (with connecting line and SEM error bars)
    daily_centred_mean = df_plot.groupby('day')['centred'].mean()
    daily_centred_sem = df_plot.groupby('day')['centred'].sem()
    ax.errorbar(daily_centred_mean.index, daily_centred_mean.values,
                yerr=daily_centred_sem.values, fmt='o-',
                color=config.accent_color, markersize=7, linewidth=2.2,
                capsize=3, zorder=10, label='Group Mean ± SEM')

    # Population trend line from the mixed-effects slope
    days_grid = np.linspace(min_day, max_day, 100)
    centroid_day = df_plot['day'].mean()
    centroid_y = daily_centred_mean.mean()
    trend_line = centroid_y + slope * (days_grid - centroid_day)
    ax.plot(days_grid, trend_line, '--', color=config.primary_color,
            alpha=0.85, linewidth=2, label='Mixed-model trend',
            zorder=9)

    remove_gridlines(ax)
    ax.set_xlabel('Day')
    _, unit = get_feature_name(feature)
    ax.set_ylabel(f'Δ from subject mean ({unit})')
    ax.set_xticks(range(min_day, max_day + 1))

    # Annotation: slope, CI, FDR p in original units
    annot = f'Slope = {slope:+.4f} {unit}/day\n95% CI [{ci_low:+.4f}, {ci_high:+.4f}]\np_FDR = {p_fdr:.3f}'
    ax.text(0.02, 0.98, annot, transform=ax.transAxes,
            fontsize=FONT['annotation'] - 1, va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor='lightgray', alpha=0.85))

    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1),
              fontsize=FONT['legend'] - 1, frameon=False)


# Plot the FDR-significant features in centred form
n_sig = len(sig_group)
print(f"Plotting {n_sig} FDR-significant features in subject-mean-centred form...")

if n_sig > 0:
    n_cols = 3
    n_rows = int(np.ceil(n_sig / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4.5 * n_rows))
    axes = axes.flatten() if n_sig > 1 else [axes]

    for i, (_, row) in enumerate(sig_group.iterrows()):
        plot_group_trend_centred(
            df, row['feature'], config.subject_col,
            slope=row['slope'],
            ci_low=row['slope_ci_low'],
            ci_high=row['slope_ci_high'],
            p_fdr=row['p_fdr'],
            ax=axes[i]
        )
        name, _ = get_feature_name(row['feature'])
        axes[i].set_title(name, fontweight='bold', fontsize=FONT['title'] - 2)

    for j in range(n_sig, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        'Group-Level Significant Features: Within-Subject Day Effect\n'
        '(subject means subtracted; dots = individual recordings, thick line = group mean ± SEM)',
        fontsize=FONT['suptitle'], fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.savefig(f"{config.output_dir}/group_significant_trends_centred.png",
                dpi=300, bbox_inches='tight')
    plt.show()

---
### Supplementary: Raw values (between-subject spread shown for completeness)

In [ ]:
n_sig = len(sig_group)
print(f"Plotting {n_sig} GROUP-level FDR-significant features (sorted by R²)...")

if n_sig > 0:
    n_cols = 3
    n_rows = int(np.ceil(n_sig / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4.5 * n_rows))
    axes = axes.flatten() if n_sig > 1 else [axes]
    
    for i, (_, row) in enumerate(sig_group.iterrows()):
        plot_group_trend_style(df, row['feature'], config.subject_col, row['r_squared'], axes[i])
        name, _ = get_feature_name(row['feature'])
        axes[i].set_title(name, fontweight='bold', fontsize=FONT['title']-2)
    
    for j in range(n_sig, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Supplementary: Raw Trends for the 9 FDR-Significant Features', fontsize=FONT['suptitle'], fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f"{config.output_dir}/supplementary_raw_trends.png", dpi=300, bbox_inches='tight')
    plt.show()

---
## 5. Machine Learning Analysis

LOSO-CV with 5 models (Random Forest, Extra Trees, XGBoost, SVR, MERF). Correlation filtering and scaling happen inside each fold to prevent leakage.

In [ ]:
# ============================================================
# COMPREHENSIVE ML ANALYSIS FUNCTION (NO LEAKAGE, RELABELLED METRICS)
# ============================================================
#
# Metric philosophy under LOSO:
#   - Pooled R^2 / RMSE / MAE: computed on out-of-fold predictions across
#     ALL subjects together. THIS is the proper cross-validated metric.
#   - Per-subject R^2: computed within each held-out subject's trajectory.
#     LOW POWER and unstable when a subject's day range is small. Reported
#     for transparency, NOT as the headline.
#   - Per-subject MAE / RMSE: well-defined per subject. Reported as
#     mean +/- SD across subjects. This is the more honest fold-level
#     summary than per-subject R^2.

from sklearn.pipeline import Pipeline


def remove_correlated_features(X_df, threshold):
    """
    Wrapper around the canonical correlation filter for use inside CV folds.
    Delegates to remove_correlated_canonical() defined in cell 11, ensuring
    the SAME filter logic is used at every call site.
    """
    features = X_df.columns.tolist()
    return remove_correlated_canonical(X_df, features, threshold)
    


def run_comprehensive_ml_analysis(df, feature_cols_initial, subject_col, day_col='day',
                                   correlation_threshold=None):
    """
    Run all ML models including MERF with LOSO-CV.

    Scaling AND feature selection happen INSIDE each CV fold to prevent
    data leakage.

    Correlation filtering rationale: With many features and a small sample,
    we have p > n. Reducing via correlation filtering improves the
    sample-to-feature ratio and reduces redundancy.
    """
    if correlation_threshold is None:
        correlation_threshold = config.correlation_threshold

    X_full_df = df[feature_cols_initial].copy()
    y = df[day_col].values
    groups = df[subject_col].values

    # Handle missing values
    valid = ~X_full_df.isna().any(axis=1) & ~np.isnan(y)
    X_full_df = X_full_df[valid].reset_index(drop=True)
    y = y[valid]
    groups = groups[valid]
    df_valid = df[valid].copy()

    logo = LeaveOneGroupOut()
    unique_subjects = np.unique(groups)
    n_subjects_cv = len(unique_subjects)

    base_models = {
        'SVR (RBF)': SVR(kernel='rbf', C=1.0, epsilon=0.1),
        'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_leaf=3, random_state=config.random_seed),
        'Extra Trees': ExtraTreesRegressor(n_estimators=100, max_depth=5, min_samples_leaf=3, random_state=config.random_seed),
    }

    if HAS_XGBOOST:
        base_models['XGBoost'] = XGBRegressor(
            n_estimators=100, max_depth=3, learning_rate=0.1,
            min_child_weight=3, subsample=0.8, colsample_bytree=0.8,
            random_state=config.random_seed, verbosity=0
        )

    results = []

    print("\n" + "=" * 78)
    print("STANDARD ML MODELS (LOSO-CV) - WITH PROPER CV (NO LEAKAGE)")
    print("=" * 78)
    print(f"Feature selection (|r| > {correlation_threshold}) happens INSIDE each fold")
    print("Scaling also happens INSIDE each fold")
    print("-" * 78)

    features_per_fold = []

    for name, base_model in base_models.items():
        y_pred = np.zeros_like(y, dtype=float)
        fold_subj_r2s = []
        fold_subj_rmses = []
        fold_subj_maes = []

        for train_idx, test_idx in logo.split(X_full_df.values, y, groups):
            X_train_df = X_full_df.iloc[train_idx]
            selected_features = remove_correlated_features(X_train_df, correlation_threshold)

            if name == list(base_models.keys())[0]:
                features_per_fold.append(len(selected_features))

            X_train = X_train_df[selected_features].values
            X_test = X_full_df.iloc[test_idx][selected_features].values
            y_train = y[train_idx]
            y_test = y[test_idx]

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            model = clone(base_model)
            model.fit(X_train_scaled, y_train)

            y_pred[test_idx] = model.predict(X_test_scaled)

            if len(y_test) > 1:
                # Per-subject R^2 (NOTE: low power, unstable - kept for transparency)
                fold_subj_r2s.append(r2_score(y_test, y_pred[test_idx]))
                # Per-subject RMSE and MAE - these ARE well-defined per fold
                fold_subj_rmses.append(np.sqrt(mean_squared_error(y_test, y_pred[test_idx])))
                fold_subj_maes.append(mean_absolute_error(y_test, y_pred[test_idx]))

        # Pooled metrics (the headline)
        overall_r2 = r2_score(y, y_pred)
        overall_rmse = np.sqrt(mean_squared_error(y, y_pred))
        overall_mae = mean_absolute_error(y, y_pred)

        results.append({
            'name': name,
            # Pooled (proper CV)
            'r2_overall': overall_r2,
            'rmse_overall': overall_rmse,
            'mae_overall': overall_mae,
            # Per-subject (transparency)
            'r2_per_subject_mean': np.mean(fold_subj_r2s),
            'r2_per_subject_std': np.std(fold_subj_r2s),
            'rmse_per_subject_mean': np.mean(fold_subj_rmses),
            'rmse_per_subject_std': np.std(fold_subj_rmses),
            'mae_per_subject_mean': np.mean(fold_subj_maes),
            'mae_per_subject_std': np.std(fold_subj_maes),
            # Back-compat aliases (deprecated names from v2 - kept so downstream code keeps working)
            'r2_mean': np.mean(fold_subj_r2s),
            'r2_std': np.std(fold_subj_r2s),
            'rmse_mean': np.mean(fold_subj_rmses),
            'rmse_std': np.std(fold_subj_rmses),
            'predictions': y_pred,
            'is_merf': False,
        })

        print(f"  {name:18s} | Pooled R2: {overall_r2:.3f} | Pooled MAE: {overall_mae:.2f}d "
              f"| Per-subj MAE: {np.mean(fold_subj_maes):.2f}+/-{np.std(fold_subj_maes):.2f}d")

    print(f"\n  Features selected per fold: {min(features_per_fold)}-{max(features_per_fold)} "
          f"(from {len(feature_cols_initial)} initial)")

    merf_random_effects = None

    if HAS_MERF:
        print("\n" + "=" * 78)
        print("MERF (Mixed-Effects Random Forest) - LOSO-CV (NO LEAKAGE)")
        print("=" * 78)
        print("MERF explicitly models: Fixed effects (RF) + Random effects (per-subject)")
        print("Feature selection and scaling happen INSIDE each fold\n")

        clusters = pd.Series(groups)

        merf_predictions = np.zeros_like(y, dtype=float)
        merf_fold_subj_r2s = []
        merf_fold_subj_rmses = []
        merf_fold_subj_maes = []
        merf_convergence_log = []  # (3b) track convergence

        pbar_merf = ProgressBar(n_subjects_cv, prefix='MERF LOSO-CV', length=30)

        for fold_i, test_subj in enumerate(unique_subjects):
            train_mask = groups != test_subj
            test_mask = groups == test_subj

            X_train_df = X_full_df[train_mask].reset_index(drop=True)
            selected_features = remove_correlated_features(X_train_df, correlation_threshold)

            X_train = X_train_df[selected_features]
            X_test = X_full_df[test_mask][selected_features].reset_index(drop=True)

            scaler = StandardScaler()
            X_train_scaled = pd.DataFrame(
                scaler.fit_transform(X_train),
                columns=selected_features
            )
            X_test_scaled = pd.DataFrame(
                scaler.transform(X_test),
                columns=selected_features
            )

            y_train = y[train_mask]
            y_test = y[test_mask]
            Z_train = np.ones((len(y_train), 1))
            Z_test = np.ones((len(y_test), 1))
            clusters_train = clusters[train_mask].reset_index(drop=True)
            clusters_test = clusters[test_mask].reset_index(drop=True)

            merf = MERF(
                fixed_effects_model=RandomForestRegressor(
                    n_estimators=100, max_depth=5, min_samples_leaf=3,
                    random_state=config.random_seed
                ),
                max_iterations=30
            )

            # Default values in case anything below fails
            n_iter = 0
            rel_change = np.nan
            converged = False
            fit_failed = False

            try:
                merf.fit(X_train_scaled, Z_train, clusters_train, y_train)

                # Extract GLL history (MERF 1.0 attribute name is 'gll_history', no trailing _)
                gll_hist = getattr(merf, 'gll_history', None)
                if gll_hist is not None and len(gll_hist) >= 2:
                    n_iter = len(gll_hist)
                    delta = abs(gll_hist[-1] - gll_hist[-2])
                    rel_change = delta / (abs(gll_hist[-2]) + 1e-9)
                    converged = bool(rel_change < 1e-3)
                elif gll_hist is not None and len(gll_hist) == 1:
                    n_iter = 1
                    converged = False  # only 1 iteration, cannot judge change

                # Always log, even if convergence info is missing
                merf_convergence_log.append({
                    'subject': test_subj,
                    'n_iter': n_iter,
                    'rel_change': rel_change,
                    'converged': converged,
                    'fit_failed': False,
                })

                y_pred_fold = merf.predict(X_test_scaled, Z_test, clusters_test)
                merf_predictions[test_mask] = y_pred_fold

            except Exception as e:
                print(f"  [WARN] MERF failed for subject {test_subj}: {str(e)[:60]}")
                merf_predictions[test_mask] = np.mean(y_train)
                merf_convergence_log.append({
                    'subject': test_subj,
                    'n_iter': n_iter,
                    'rel_change': rel_change,
                    'converged': False,
                    'fit_failed': True,
                })
                fit_failed = True
                continue

            if len(y_test) > 1:
                merf_fold_subj_r2s.append(r2_score(y_test, y_pred_fold))
                merf_fold_subj_rmses.append(np.sqrt(mean_squared_error(y_test, y_pred_fold)))
                merf_fold_subj_maes.append(mean_absolute_error(y_test, y_pred_fold))

            pbar_merf.update(suffix=f'[Subject {test_subj}]')

        merf_r2 = r2_score(y, merf_predictions)
        merf_rmse = np.sqrt(mean_squared_error(y, merf_predictions))
        merf_mae = mean_absolute_error(y, merf_predictions)

        results.append({
            'name': 'MERF',
            'r2_overall': merf_r2,
            'rmse_overall': merf_rmse,
            'mae_overall': merf_mae,
            'r2_per_subject_mean': np.mean(merf_fold_subj_r2s),
            'r2_per_subject_std': np.std(merf_fold_subj_r2s),
            'rmse_per_subject_mean': np.mean(merf_fold_subj_rmses),
            'rmse_per_subject_std': np.std(merf_fold_subj_rmses),
            'mae_per_subject_mean': np.mean(merf_fold_subj_maes),
            'mae_per_subject_std': np.std(merf_fold_subj_maes),
            # Back-compat
            'r2_mean': np.mean(merf_fold_subj_r2s),
            'r2_std': np.std(merf_fold_subj_r2s),
            'rmse_mean': np.mean(merf_fold_subj_rmses),
            'rmse_std': np.std(merf_fold_subj_rmses),
            'predictions': merf_predictions,
            'is_merf': True,
        })

        print(f"\n  {'MERF':18s} | Pooled R2: {merf_r2:.3f} | Pooled MAE: {merf_mae:.2f}d "
              f"| Per-subj MAE: {np.mean(merf_fold_subj_maes):.2f}+/-{np.std(merf_fold_subj_maes):.2f}d")

        # Convergence summary (uses dict-of-dicts log; reports per-fold detail)
        n_total = len(merf_convergence_log)
        n_failed = sum(1 for r in merf_convergence_log if r['fit_failed'])
        globals()['MERF_CONVERGENCE_LOG'] = list(merf_convergence_log)
        n_conv = sum(1 for r in merf_convergence_log if r['converged'] and not r['fit_failed'])
        n_unconv = sum(1 for r in merf_convergence_log if not r['converged'] and not r['fit_failed'])
        print(f"  MERF convergence (criterion: rel GLL change < 1e-3 between last two iterations):")
        print(f"    Converged:  {n_conv}/{n_total} folds")
        print(f"    Did not:    {n_unconv}/{n_total} folds")
        if n_failed > 0:
            print(f"    Fit failed: {n_failed}/{n_total} folds")
        # Per-fold breakdown
        print(f"    Per-fold detail:")
        for r in merf_convergence_log:
            status = "FAILED" if r['fit_failed'] else ("converged" if r['converged'] else "not conv")
            rc = f"{r['rel_change']:.2e}" if not (r['rel_change'] is None or (isinstance(r['rel_change'], float) and np.isnan(r['rel_change']))) else "n/a"
            print(f"      Subject {r['subject']}: iters={r['n_iter']:>3d}, rel_change={rc:>10s}, {status}")

        # Train final MERF for random effects (on all data, for INTERPRETATION ONLY)
        print("\n  Training final MERF model for random effects estimation...")
        print("  (FYI: this is fit on the full dataset and is NOT used for predictive metrics.)")
        selected_features_all = remove_correlated_features(X_full_df, correlation_threshold)
        scaler_final = StandardScaler()
        X_scaled_final = pd.DataFrame(
            scaler_final.fit_transform(X_full_df[selected_features_all]),
            columns=selected_features_all
        )
        Z = np.ones((len(y), 1))

        merf_final = MERF(
            fixed_effects_model=RandomForestRegressor(
                n_estimators=100, max_depth=5, min_samples_leaf=3,
                random_state=config.random_seed
            ),
            max_iterations=30
        )
        merf_final.fit(X_scaled_final, Z, clusters, y)
        merf_random_effects = merf_final.trained_b
        print("  [OK] Random effects estimated")

    # Feature importance (on full data with correlation filtering - INTERPRETATION ONLY)
    print("\n  Computing feature importance (for interpretation)...")
    selected_features_all = remove_correlated_features(X_full_df, correlation_threshold)
    scaler_fi = StandardScaler()
    X_scaled_fi = scaler_fi.fit_transform(X_full_df[selected_features_all])
    rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_leaf=3, random_state=config.random_seed)
    rf_model.fit(X_scaled_fi, y)
    feature_importance = pd.DataFrame({
        'feature': selected_features_all,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    print(f"  [OK] Feature importance computed ({len(selected_features_all)} features)")

    return results, y, groups, df_valid, feature_importance, unique_subjects, merf_random_effects


In [ ]:
# ============================================================
# COMPREHENSIVE VISUALIZATION FUNCTION
# ============================================================

def plot_comprehensive_results(results, y_actual, groups, df_valid, feature_importance,
                                unique_subjects, merf_random_effects, subject_col):
    """
    Create comprehensive visualization with all models including MERF.

    Notes
    -----
    - The "Per-Subject Prediction Quality" panel shows per-trajectory R^2.
      Under LOSO, each test fold IS one subject's whole trajectory, so
      this is per-subject within-trajectory R^2 (how well the model orders
      that subject's recordings by predicted day). It is NOT cross-validated
      R^2 in the conventional sense.
    """
    # Find best model
    best_idx = np.argmax([r['r2_overall'] for r in results])
    best_model_name = results[best_idx]['name']
    best_predictions = results[best_idx]['predictions']

    # Prepare data
    df_plot = df_valid.copy()
    df_plot['progression_score'] = best_predictions
    days_unique = sorted(df_plot['day'].unique())
    daily_mean = df_plot.groupby('day')['progression_score'].mean()
    daily_count = df_plot.groupby('day')['progression_score'].count()
    residuals = y_actual - best_predictions
    subjs = sorted(df_plot[subject_col].unique())

    # Get day range for consistent axes
    min_day = int(min(days_unique))
    max_day = int(max(days_unique))
    all_days = list(range(min_day, max_day + 1))

    # Colors
    colors = get_blue_palette(len(subjs) + 2)[2:]

    print("\n  Generating visualization...")

    fig, axes = plt.subplots(3, 3, figsize=(16, 13))

    # [0,0] Boxplots by Day
    ax = axes[0, 0]
    box_data = [df_plot[df_plot['day']==d]['progression_score'].values for d in days_unique]
    bp = ax.boxplot(box_data, positions=days_unique, widths=0.6, patch_artist=True,
                    medianprops=dict(color='black', linewidth=1.5),
                    boxprops=dict(facecolor='white', edgecolor='black', linewidth=1.5),
                    whiskerprops=dict(color='black', linewidth=1.5),
                    capprops=dict(color='black', linewidth=1.5),
                    flierprops=dict(marker='o', markerfacecolor='none', markeredgecolor='black', markersize=4))

    # WEIGHTED polyfit (3f): weight by per-day count so heavier-sampled days dominate appropriately
    z = np.polyfit(daily_mean.index, daily_mean.values, 1, w=daily_count.values)
    p = np.poly1d(z)
    x_line = np.linspace(min(days_unique), max(days_unique), 100)
    ax.plot(x_line, p(x_line), '-', color=PRIMARY_COLOR, linewidth=2.5, label='Weighted trend')
    ax.set_xlabel('Day')
    ax.set_ylabel('Predicted Day (Progression Score)')
    ax.set_title('Progression Score by Day', fontweight='bold')
    ax.set_xticks(all_days)
    ax.set_xticklabels(all_days)
    ax.set_xlim(min_day - 0.5, max_day + 0.5)
    ax.legend(fontsize=8, frameon=False)
    remove_gridlines(ax)

    # [0,1] Individual Trajectories
    ax = axes[0, 1]
    for i, subj in enumerate(subjs):
        subj_data = df_plot[df_plot[subject_col] == subj].sort_values('day')
        ax.plot(subj_data['day'], subj_data['progression_score'], 'o-',
               color=colors[i % len(colors)], alpha=0.7, markersize=5, linewidth=1.5, label=f'{subj}')
    ax.plot(daily_mean.index, daily_mean.values, 's-', color='black',
            markersize=7, linewidth=2.5, label='Group Mean', zorder=10)
    ax.set_xlabel('Day')
    ax.set_ylabel('Predicted Day (Progression Score)')
    ax.set_title('Individual Trajectories', fontweight='bold')
    ax.set_xticks(all_days)
    ax.set_xticklabels(all_days)
    ax.set_xlim(min_day - 0.5, max_day + 0.5)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7, frameon=False)
    remove_gridlines(ax)

    # [0,2] Prediction Accuracy with MAE
    ax = axes[0, 2]
    ax.scatter(y_actual, best_predictions, alpha=0.6, color=PRIMARY_COLOR,
               s=40, edgecolor='white', linewidth=0.5)
    ax.plot([min_day, max_day], [min_day, max_day], '--', color='gray', linewidth=1.5)
    r2 = results[best_idx]['r2_overall']
    rmse = results[best_idx]['rmse_overall']
    mae = results[best_idx].get('mae_overall', np.mean(np.abs(y_actual - best_predictions)))
    ax.text(0.95, 0.05,
            f'Pooled R$^2$ = {r2:.3f}\nRMSE = {rmse:.2f} days\nMAE = {mae:.2f} days',
            transform=ax.transAxes,
            fontsize=10, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.set_xlabel('Actual Day')
    ax.set_ylabel('Predicted Day')
    ax.set_title(f'Prediction Accuracy ({best_model_name})', fontweight='bold')
    ax.set_xticks(all_days)
    ax.set_yticks(all_days)
    ax.set_xticklabels(all_days)
    ax.set_yticklabels(all_days)
    ax.set_xlim(min_day - 0.5, max_day + 0.5)
    ax.set_ylim(min_day - 0.5, max_day + 0.5)
    remove_gridlines(ax)

    # [1,0] Model RMSE Comparison (per-subject mean +/- SD)
    ax = axes[1, 0]
    model_names = [r['name'] for r in results]
    y_pos = np.arange(len(model_names))
    rmse_vals = [r['rmse_per_subject_mean'] for r in results]
    rmse_stds = [r['rmse_per_subject_std'] for r in results]

    for i, (rmse_val, std) in enumerate(zip(rmse_vals, rmse_stds)):
        ax.errorbar(rmse_val, i, xerr=std, fmt='o', markersize=8, capsize=4,
                    color=PRIMARY_COLOR, ecolor=PRIMARY_COLOR, elinewidth=1.5,
                    markeredgecolor='white', markeredgewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(model_names)
    ax.set_xlabel('Per-subject RMSE (mean +/- SD)')
    ax.set_title('Model RMSE (per-subject)', fontweight='bold')
    ax.invert_yaxis()
    remove_gridlines(ax)

    # [1,1] Model MAE Comparison (well-defined per fold, unlike R^2)
    ax = axes[1, 1]
    mae_vals = [r['mae_per_subject_mean'] for r in results]
    mae_stds = [r['mae_per_subject_std'] for r in results]

    for i, (mae_val, std) in enumerate(zip(mae_vals, mae_stds)):
        ax.errorbar(mae_val, i, xerr=std, fmt='o', markersize=8, capsize=4,
                    color=PRIMARY_COLOR, ecolor=PRIMARY_COLOR, elinewidth=1.5,
                    markeredgecolor='white', markeredgewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(model_names)
    ax.set_xlabel('Per-subject MAE (mean +/- SD, days)')
    ax.set_title('Model MAE (per-subject)', fontweight='bold')
    ax.invert_yaxis()
    remove_gridlines(ax)

    # [1,2] Feature Importance
    ax = axes[1, 2]
    top_n = min(10, len(feature_importance))
    top_feat = feature_importance.head(top_n).iloc[::-1]
    y_pos_feat = np.arange(top_n)

    ax.barh(y_pos_feat, top_feat['importance'], color=PRIMARY_COLOR, edgecolor='none', height=0.7)
    ax.set_yticks(y_pos_feat)
    ax.set_yticklabels([get_feature_name(f, 25)[0] for f in top_feat['feature']], fontsize=9)
    ax.set_xlabel('Importance')
    ax.set_title('Feature Importance (RF, full data)', fontweight='bold')
    remove_gridlines(ax)

    # [2,0] Residual Distribution
    ax = axes[2, 0]
    ax.hist(residuals, bins=15, color=PRIMARY_COLOR, edgecolor='white', alpha=0.8)
    ax.axvline(x=0, color='black', linewidth=1.5, linestyle='--')
    mean_res = np.mean(residuals)
    std_res = np.std(residuals)
    ax.set_xlabel('Residual (Actual - Predicted)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Residual Distribution\n(Mean={mean_res:.2f}, SD={std_res:.2f})', fontweight='bold')
    remove_gridlines(ax)

    # [2,1] Residuals vs Predicted using LOWESS (replaces UnivariateSpline, 3i)
    ax = axes[2, 1]
    ax.scatter(best_predictions, residuals, alpha=0.6, color=PRIMARY_COLOR,
               s=40, edgecolor='white', linewidth=0.5, zorder=3)
    ax.axhline(y=0, color='black', linewidth=1.5, linestyle='--', zorder=2)

    try:
        from statsmodels.nonparametric.smoothers_lowess import lowess
        # frac controls smoothing; 0.5 is moderate
        sm_curve = lowess(residuals, best_predictions, frac=0.5, return_sorted=True)
        x_smooth = sm_curve[:, 0]
        trend_smooth = sm_curve[:, 1]

        # Per-bin SD for the band (rolling-window approach)
        sort_idx = np.argsort(best_predictions)
        x_sorted = best_predictions[sort_idx]
        res_sorted = residuals[sort_idx]
        n_bins = max(5, min(10, len(x_sorted) // 6))
        bin_edges = np.linspace(x_sorted.min(), x_sorted.max(), n_bins + 1)
        bin_stds, bin_centers = [], []
        for j in range(n_bins):
            mask = (x_sorted >= bin_edges[j]) & (x_sorted < bin_edges[j+1])
            if mask.sum() > 1:
                bin_stds.append(np.std(res_sorted[mask]))
                bin_centers.append((bin_edges[j] + bin_edges[j+1]) / 2)
        if len(bin_centers) > 2:
            std_smooth = np.interp(x_smooth, bin_centers, bin_stds)
        else:
            std_smooth = np.full_like(x_smooth, np.std(residuals))

        ax.fill_between(x_smooth, trend_smooth - std_smooth, trend_smooth + std_smooth,
                        color='salmon', alpha=0.3, zorder=1, label='+/-1 SD band')
        ax.plot(x_smooth, trend_smooth, '-', color='salmon', linewidth=2.5, alpha=0.9,
                zorder=2, label='LOWESS trend')
    except Exception as e:
        # Fallback: simple polyfit
        z = np.polyfit(best_predictions, residuals, 2)
        x_smooth = np.linspace(best_predictions.min(), best_predictions.max(), 100)
        ax.plot(x_smooth, np.polyval(z, x_smooth), '-', color='salmon', linewidth=2.5,
                label='Polynomial fallback')

    ax.set_xlabel('Predicted Day')
    ax.set_ylabel('Residual')
    ax.set_title('Residuals vs Predicted', fontweight='bold')

    pred_min = int(np.floor(best_predictions.min()))
    pred_max = int(np.ceil(best_predictions.max()))
    pred_days = list(range(pred_min, pred_max + 1))
    ax.set_xticks(pred_days)
    ax.set_xticklabels(pred_days)

    ax.legend(loc='upper right', fontsize=7, frameon=True)
    remove_gridlines(ax)

    # [2,2] Per-Subject Prediction Quality (R^2 on each subject's trajectory)
    # Note (1e): This is per-trajectory R^2, NOT cross-validated R^2.
    # Negative values just mean the model orders that subject's days worse than the mean.
    ax = axes[2, 2]
    subject_r2s, subject_names, subject_indices = [], [], []
    for i, subj in enumerate(subjs):
        mask = df_plot[subject_col] == subj
        y_subj = df_plot.loc[mask, 'day'].values
        pred_subj = df_plot.loc[mask, 'progression_score'].values
        if len(y_subj) > 1:
            r2_subj = r2_score(y_subj, pred_subj)
            subject_r2s.append(r2_subj)
            subject_names.append(str(subj))
            subject_indices.append(i)

    y_pos_subj = np.arange(len(subject_names))
    # Guard against length mismatch (3d): use subject_indices, not raw enumerate
    bar_colors_subj = [
        colors[subject_indices[k] % len(colors)] if subject_r2s[k] >= 0 else NEGATIVE_COLOR
        for k in range(len(subject_r2s))
    ]

    ax.barh(y_pos_subj, subject_r2s, color=bar_colors_subj, edgecolor='none', height=0.7)
    ax.axvline(x=0, color='black', linewidth=1, linestyle='-')
    ax.set_yticks(y_pos_subj)
    ax.set_yticklabels(subject_names, fontsize=9)
    ax.set_xlabel('Per-trajectory R$^2$ (NOT cross-validated R$^2$)')
    ax.set_title('Per-Subject Trajectory Fit\n(model ordering quality, not CV)', fontweight='bold')
    ax.invert_yaxis()
    remove_gridlines(ax)

    plt.suptitle(f'Machine Learning Analysis - Model Comparison\n(LOSO-CV, Best by pooled R$^2$: {best_model_name})',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.subplots_adjust(left=0.12)
    plt.savefig(f"{config.output_dir}/ml_analysis_comprehensive.png", dpi=300, bbox_inches='tight')
    plt.show()

    print("  [OK] Visualization saved")

    return results, subject_r2s


In [ ]:
# ============================================================
# RUN COMPREHENSIVE ML ANALYSIS
# ============================================================

import logging
logging.getLogger('merf').setLevel(logging.WARNING)


print("=" * 78)
print("COMPREHENSIVE ML ANALYSIS")
print("=" * 78)
print(f"\nStarting with {len(feature_cols_initial)} features")
print(f"Correlation filtering (|r| > {config.correlation_threshold}) applied INSIDE each CV fold")
print("This prevents data leakage while reducing dimensionality (p > n problem)")
print("LOSO-CV = proper handling of subject clustering")
print("-" * 78)

ml_results, y_actual, groups, df_valid, feature_importance, unique_subjects, merf_random_effects = \
    run_comprehensive_ml_analysis(
        df,
        feature_cols_initial,
        config.subject_col,
        'day',
        correlation_threshold=config.correlation_threshold
    )


# ============================================================
# PRINT MODEL PERFORMANCE TABLE (with proper labels)
# ============================================================

print("\n" + "=" * 92)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 92)
print()
print("Pooled = computed across all out-of-fold predictions (proper CV metric)")
print("Per-subj = mean +/- SD across LOSO folds (one fold = one subject)")
print()

print(f"{'Model':<15} {'Pooled R2':>10} {'Pooled MAE':>12} {'Pooled RMSE':>12} "
      f"{'Per-subj MAE':>16} {'Per-subj RMSE':>16}")
print("-" * 92)

best_idx = np.argmax([r['r2_overall'] for r in ml_results])

for i, r in enumerate(ml_results):
    marker = " *" if i == best_idx else ""
    mae_str = f"{r['mae_per_subject_mean']:.2f}+/-{r['mae_per_subject_std']:.2f}"
    rmse_str = f"{r['rmse_per_subject_mean']:.2f}+/-{r['rmse_per_subject_std']:.2f}"
    print(f"{r['name']:<15} {r['r2_overall']:>10.3f} {r['mae_overall']:>11.2f}d "
          f"{r['rmse_overall']:>11.2f}d {mae_str:>16} {rmse_str:>16}{marker}")

print("-" * 92)
print("* = Best model by pooled R2")
print()
print("[NOTE] Per-subject R2 is intentionally NOT shown as a primary metric.")
print("       Under LOSO it is per-trajectory (one subject per fold) and can be")
print("       wildly negative or unstable when a subject's day range is small.")
print("       It is preserved in the results dict as r2_per_subject_mean/std for")
print("       transparency, and visualized in the per-subject panel.")
print()

best = ml_results[best_idx]
print(f"Best model by pooled R2: {best['name']}")
print(f"  - Pooled R2:   {best['r2_overall']:.3f}")
print(f"  - Pooled MAE:  {best['mae_overall']:.2f} days")
print(f"  - Pooled RMSE: {best['rmse_overall']:.2f} days")
print(f"  - Per-subject MAE:  {best['mae_per_subject_mean']:.2f} +/- {best['mae_per_subject_std']:.2f} days")
print(f"  - Per-subject RMSE: {best['rmse_per_subject_mean']:.2f} +/- {best['rmse_per_subject_std']:.2f} days")


In [ ]:
# Plot results
plot_comprehensive_results(
    ml_results, y_actual, groups, df_valid, feature_importance, 
    unique_subjects, merf_random_effects, config.subject_col
)


---
## 6. Variance Decomposition for the 9 Headline Features

For each FDR-significant feature, decompose total variance into between-subject (Individual), day effect (State), and unexplained (Residual) components, with cluster-bootstrap 95% confidence intervals on the State component.

In [ ]:
# ============================================================
# VARIANCE DECOMPOSITION BY FEATURE (with bootstrap CIs - 2a)
# ============================================================
# Uses only the features that passed correlation filtering.
# Bootstrap is over SUBJECTS (cluster bootstrap) so CIs reflect the
# limited number of clusters, which is the dominant source of uncertainty.

import statsmodels.formula.api as smf

def _fit_variance_components(data, feature, subject_col, day_col):
    """Fit a random-intercept mixed model and return prop. variance components."""
    model = smf.mixedlm(
        f'Q("{feature}") ~ {day_col}',
        data=data,
        groups=data[subject_col]
    )
    fit = model.fit(disp=False, method='lbfgs')

    var_subject = fit.cov_re.iloc[0, 0] if hasattr(fit.cov_re, 'iloc') else float(fit.cov_re)
    var_residual = fit.scale

    y = data[feature].values
    total_var = np.var(y, ddof=1)
    fixed_pred = fit.fe_params['Intercept'] + fit.fe_params[day_col] * data[day_col].values
    var_fixed = np.var(fixed_pred, ddof=1)

    prop_individual = var_subject / total_var
    prop_state = var_fixed / total_var
    prop_unexplained = var_residual / total_var

    total_prop = prop_individual + prop_state + prop_unexplained
    return (
        prop_individual / total_prop * 100,
        prop_state / total_prop * 100,
        prop_unexplained / total_prop * 100,
        fit.pvalues[day_col],
        fit.fe_params[day_col],
        fit.converged,
    )


def variance_decomposition_analysis(df, features, subject_col, day_col='day',
                                     n_bootstrap=200, random_state=None):
    """
    Decompose variance for each feature into:
    - Between-subject (individual differences)
    - Fixed effect of day (isolation/state effect)
    - Residual (unexplained)

    Bootstrap CIs computed by resampling SUBJECTS with replacement
    (cluster bootstrap), which is the appropriate uncertainty for
    repeated-measures data with few clusters.
    """
    rng = np.random.default_rng(random_state if random_state is not None else config.random_seed)

    results = []
    failed_features = []
    skipped_features = []
    nonconverged = []

    print("=" * 78)
    print("VARIANCE DECOMPOSITION ANALYSIS (with cluster-bootstrap CIs)")
    print("=" * 78)
    print(f"\nAnalyzing {len(features)} features (correlation filtered, |r| < {config.correlation_threshold})")
    print(f"Bootstrap: {n_bootstrap} subject-level resamples per feature")
    print("-" * 78)

    subjects_unique = df[subject_col].unique()

    for feature in tqdm(features, desc="Analyzing features"):
        data = df[[subject_col, day_col, feature]].dropna()

        if len(data) < 10 or data[subject_col].nunique() < 3:
            skipped_features.append((feature, f"Insufficient data: n={len(data)}, subjects={data[subject_col].nunique()}"))
            continue

        try:
            prop_ind, prop_state, prop_unex, p_day, slope, conv = _fit_variance_components(
                data, feature, subject_col, day_col
            )
            if not conv:
                nonconverged.append(feature)

            # Cluster bootstrap (resample subjects with replacement)
            boot_ind, boot_state, boot_unex = [], [], []
            for b in range(n_bootstrap):
                sampled_subj = rng.choice(subjects_unique, size=len(subjects_unique), replace=True)
                boot_data_parts = []
                for new_id, s in enumerate(sampled_subj):
                    chunk = data[data[subject_col] == s].copy()
                    if len(chunk) == 0:
                        continue
                    chunk[subject_col] = f"_b{new_id}"  # unique pseudo-IDs to avoid duplicate-cluster issues
                    boot_data_parts.append(chunk)
                if not boot_data_parts:
                    continue
                boot_data = pd.concat(boot_data_parts, ignore_index=True)
                if boot_data[subject_col].nunique() < 3:
                    continue
                try:
                    bi, bs, bu, _, _, _ = _fit_variance_components(boot_data, feature, subject_col, day_col)
                    boot_ind.append(bi)
                    boot_state.append(bs)
                    boot_unex.append(bu)
                except Exception:
                    continue

            def ci(x):
                if len(x) < 10:
                    return (np.nan, np.nan)
                return (float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5)))

            ind_lo, ind_hi = ci(boot_ind)
            st_lo, st_hi = ci(boot_state)
            un_lo, un_hi = ci(boot_unex)

            results.append({
                'feature': feature,
                # primary names (used by visualization)
                'prop_individual': prop_ind,
                'prop_state': prop_state,
                'prop_unexplained': prop_unex,
                # bootstrap CIs
                'prop_individual_ci_low': ind_lo,
                'prop_individual_ci_high': ind_hi,
                'prop_state_ci_low': st_lo,
                'prop_state_ci_high': st_hi,
                'prop_unexplained_ci_low': un_lo,
                'prop_unexplained_ci_high': un_hi,
                'n_bootstrap_valid': len(boot_state),
                # day effect
                'p_day': p_day,
                'slope': slope,
                # ALIASES for the PDF report (3e fix - the PDF expected these names)
                'individual_var_pct': prop_ind,
                'state_var_pct': prop_state,
                'unexplained_var_pct': prop_unex,
            })

        except Exception as e:
            failed_features.append((feature, str(e)))
            continue

    results_df = pd.DataFrame(results)

    if skipped_features:
        print(f"\n[INFO] Skipped {len(skipped_features)} features due to insufficient data")
    if nonconverged:
        print(f"[WARN] {len(nonconverged)} features had non-converged mixed model fits")
    if failed_features:
        print(f"[WARN] Failed to fit {len(failed_features)} features:")
        for feat, err in failed_features[:5]:
            fname, _ = get_feature_name(feat)
            print(f"    - {fname}: {err[:50]}...")
        if len(failed_features) > 5:
            print(f"    ... and {len(failed_features) - 5} more")

    if len(results_df) == 0:
        print("No features could be analyzed!")
        return None

    results_df = results_df.sort_values('prop_state', ascending=False)

    from statsmodels.stats.multitest import multipletests
    _, p_fdr, _, _ = multipletests(results_df['p_day'].values, method='fdr_bh')
    results_df['p_fdr'] = p_fdr
    results_df['sig_fdr'] = p_fdr < 0.05

    # ============================================================
    # PRINT
    # ============================================================
    print(f"\n{'Feature':<33} {'Indiv (95%CI)':>20} {'State (95%CI)':>20} {'Unexpl. (95%CI)':>20} {'p(FDR)':>8}")
    print("-" * 110)
    for _, row in results_df.iterrows():
        name, _ = get_feature_name(row['feature'])
        name = name[:31]
        sig = "*" if row['sig_fdr'] else ""
        ind_str = f"{row['prop_individual']:>5.1f} [{row['prop_individual_ci_low']:.1f},{row['prop_individual_ci_high']:.1f}]"
        st_str = f"{row['prop_state']:>5.1f} [{row['prop_state_ci_low']:.1f},{row['prop_state_ci_high']:.1f}]"
        un_str = f"{row['prop_unexplained']:>5.1f} [{row['prop_unexplained_ci_low']:.1f},{row['prop_unexplained_ci_high']:.1f}]"
        print(f"{name:<33} {ind_str:>20} {st_str:>20} {un_str:>20} {row['p_fdr']:>7.3f}{sig}")

    n_sig = results_df['sig_fdr'].sum()
    print(f"\n* = FDR-significant (p < 0.05), n = {n_sig}/{len(results_df)}")
    print("CIs are cluster-bootstrap (subject-level resampling). Wide CIs indicate")
    print("the variance component is poorly identified by the small number of clusters.")

    # ============================================================
    # VISUALIZATION (with CI whiskers on STATE component)
    # ============================================================
    print("\n  Generating visualization...")

    color_individual = PRIMARY_COLOR
    color_state = 'salmon'
    color_unexplained = '#d3d3d3'

    n_features = len(results_df)
    fig_height = max(4, n_features * 0.5)

    fig, ax = plt.subplots(figsize=(11, fig_height))

    feature_names = [get_feature_name(f, 30)[0] for f in results_df['feature']]
    y_pos = np.arange(n_features)

    # Stacked bars (point estimates)
    ax.barh(y_pos, results_df['prop_individual'].values,
            color=color_individual, height=0.65, label='Individual')
    ax.barh(y_pos, results_df['prop_state'].values,
            left=results_df['prop_individual'].values,
            color=color_state, height=0.65, label='State (Isolation)')
    ax.barh(y_pos, results_df['prop_unexplained'].values,
            left=(results_df['prop_individual'] + results_df['prop_state']).values,
            color=color_unexplained, height=0.65, label='Unexplained')

    # Add CI whiskers on the STATE component (the most-interpreted one)
    state_starts = results_df['prop_individual'].values
    state_centers = state_starts + results_df['prop_state'].values
    state_ci_lo = state_starts + results_df['prop_state_ci_low'].values
    state_ci_hi = state_starts + results_df['prop_state_ci_high'].values
    for i in range(n_features):
        if not (np.isnan(state_ci_lo[i]) or np.isnan(state_ci_hi[i])):
            ax.plot([state_ci_lo[i], state_ci_hi[i]], [i, i],
                    color='black', linewidth=1.0, alpha=0.5)
            ax.plot([state_ci_lo[i]], [i], '|', color='black', markersize=6, alpha=0.5)
            ax.plot([state_ci_hi[i]], [i], '|', color='black', markersize=6, alpha=0.5)

    for i, (_, row) in enumerate(results_df.iterrows()):
        if row['sig_fdr']:
            ax.text(106, i, '*', ha='left', va='center', fontsize=12,
                    fontweight='bold', color='salmon')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(feature_names, fontsize=10)
    ax.set_xlabel('Variance Explained (%) - whiskers = 95% CI on STATE component', fontsize=11)
    ax.set_title('Variance Decomposition by Feature (with cluster-bootstrap CIs)',
                 fontweight='bold', fontsize=13, pad=25)
    ax.set_xlim(0, 112)
    ax.invert_yaxis()

    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=3,
              fontsize=10, frameon=False)

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    ax.text(0.01, -0.02, '* = FDR-significant (p < 0.05)',
            transform=ax.transAxes, ha='left', va='top',
            fontsize=9, fontstyle='italic', color='gray')

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.savefig(f"{config.output_dir}/variance_decomposition.png", dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()

    print(f"\n  [OK] Saved to {config.output_dir}/variance_decomposition.png")

    results_df.to_csv(f"{config.output_dir}/variance_decomposition.csv", index=False)

    return results_df


# ============================================================
# RUN ANALYSIS
# ============================================================
variance_results = variance_decomposition_analysis(
    df, sig_group['feature'].tolist(), config.subject_col, 'day',
    n_bootstrap=200,  # adjust if runtime is a concern
)


---
## Results manifest

Writes `results_manifest.json`, the single source of every number reported in the manuscript.

In [ ]:
# ============================================================
# RESULTS MANIFEST (single source of truth for reported numbers)
# ============================================================
# Every number reported in the manuscript is registered here with a key.
# The manuscript is checked against results_manifest.json.

import hashlib, json, os, platform, datetime, importlib.metadata as im
from sklearn.model_selection import LeaveOneGroupOut

MANIFEST = {}

def reg(key, value, decimals, unit="", label=""):
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, (list, tuple)):
        value = [v.item() if hasattr(v, "item") else v for v in value]
    MANIFEST[key] = {"value": value, "decimals": decimals, "unit": unit, "label": label}

# Cohort
per_subj = df.groupby(config.subject_col).size()
reg("n_recordings", n_observations, 0, "", "Recordings analysed")
reg("n_participants", n_subjects, 0, "", "Participants")
reg("recordings_per_participant_mean", n_observations / n_subjects, 1, "", "Mean recordings per participant")
reg("recordings_per_participant_min", int(per_subj.min()), 0)
reg("recordings_per_participant_max", int(per_subj.max()), 0)
reg("day_min", int(df['day'].min()), 0, "day")
reg("day_max", int(df['day'].max()), 0, "day")

# Feature selection
reg("n_features_initial", n_features_initial, 0)
reg("n_features_removed_filter", n_features_initial - len(remove_correlated_canonical(df, feature_cols_initial, config.correlation_threshold)), 0)
reg("n_features_retained_filter", len(remove_correlated_canonical(df, feature_cols_initial, config.correlation_threshold)), 0)
reg("n_features_rescued", len(RESCUED_FEATURES), 0)
reg("n_features_analysed", len(feature_cols), 0)

# Trend analysis
reg("n_sig_raw", int(group_trends['sig_raw'].sum()), 0)
reg("n_sig_fdr", int(group_trends['sig_fdr'].sum()), 0)
reg("n_models_not_converged", int((~group_trends['converged']).sum()), 0)
reg("n_models_ols_fallback", int((group_trends['method'] == 'ols_fallback').sum()), 0)
for _, r in sig_group.iterrows():
    k = r['feature']
    reg(f"slope[{k}]", float(r['slope']), 4)
    reg(f"slope_ci[{k}]", [float(r['slope_ci_low']), float(r['slope_ci_high'])], 4)
    reg(f"p_fdr[{k}]", float(r['p_fdr']), 3)
    reg(f"r2[{k}]", float(r['r_squared']), 3)
    reg(f"direction[{k}]", r['direction'], 0)
reg("largest_r2_feature", sig_group.iloc[0]['feature'], 0)

# Variance decomposition
vr = variance_results
reg("state_pct_min", float(vr['prop_state'].min()), 1, "%")
reg("state_pct_max", float(vr['prop_state'].max()), 1, "%")
reg("state_pct_min_feature", vr.loc[vr['prop_state'].idxmin(), 'feature'], 0)
reg("state_pct_max_feature", vr.loc[vr['prop_state'].idxmax(), 'feature'], 0)
nonzero_ind = vr[vr['prop_individual'] > 0.05]
zero_ind = vr[vr['prop_individual'] <= 0.05]
reg("n_features_individual_zero", len(zero_ind), 0)
reg("features_individual_zero", sorted(zero_ind['feature'].tolist()), 0)
largest_term = vr[vr['prop_individual'] >= vr[['prop_state', 'prop_unexplained']].max(axis=1)]
reg("individual_pct_min_where_largest", float(largest_term['prop_individual'].min()), 0, "%")
reg("individual_pct_max_where_largest", float(largest_term['prop_individual'].max()), 0, "%")
reg("features_individual_largest", sorted(largest_term['feature'].tolist()), 0)
reg("n_bootstrap", 200, 0)
for _, r in vr.iterrows():
    reg(f"state_pct[{r['feature']}]", float(r['prop_state']), 1, "%")
    reg(f"state_pct_ci[{r['feature']}]", [float(r['prop_state_ci_low']), float(r['prop_state_ci_high'])], 1, "%")
    reg(f"individual_pct[{r['feature']}]", float(r['prop_individual']), 1, "%")

# Machine learning
for r in ml_results:
    n = r['name']
    reg(f"r2_pooled[{n}]", float(r['r2_overall']), 3)
    reg(f"mae_pooled[{n}]", float(r['mae_overall']), 2, "days")
    reg(f"rmse_pooled[{n}]", float(r['rmse_overall']), 2, "days")
    reg(f"mae_per_subject_mean[{n}]", float(r['mae_per_subject_mean']), 2, "days")
    reg(f"mae_per_subject_sd[{n}]", float(r['mae_per_subject_std']), 2, "days")
best = ml_results[int(np.argmax([r['r2_overall'] for r in ml_results]))]
reg("best_model", best['name'], 0)
reg("model_ranking", [r['name'] for r in sorted(ml_results, key=lambda r: -r['r2_overall'])], 0)
resid = y_actual - best['predictions']
reg("resid_mean", float(np.mean(resid)), 2, "days")
reg("resid_sd", float(np.std(resid)), 2, "days")  # population SD, as in Figure 3

X_ = df[feature_cols_initial]
valid_ = ~X_.isna().any(axis=1)
Xv, gv = X_[valid_].reset_index(drop=True), df.loc[valid_, config.subject_col].values
fpf = [len(remove_correlated_canonical(Xv.iloc[tr], feature_cols_initial, config.correlation_threshold))
       for tr, _ in LeaveOneGroupOut().split(Xv, groups=gv)]
reg("features_per_fold_min", min(fpf), 0)
reg("features_per_fold_max", max(fpf), 0)
reg("n_complete_recordings_ml", int(valid_.sum()), 0)

log = globals().get('MERF_CONVERGENCE_LOG', [])
reg("merf_folds_converged", sum(1 for r in log if r['converged'] and not r['fit_failed']), 0)
reg("merf_folds_total", len(log), 0)

top10 = feature_importance.head(10)['feature'].tolist()
reg("importance_top10", top10, 0)
reg("importance_max", float(feature_importance['importance'].max()), 2)
reg("n_top10_in_headline", len(set(top10) & set(sig_group['feature'])), 0)
reg("top10_in_headline", sorted(set(top10) & set(sig_group['feature'])), 0)

# Provenance
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

FIGURE_FILES = [f"{config.output_dir}/group_significant_trends_centred.png",
                f"{config.output_dir}/variance_decomposition.png",
                f"{config.output_dir}/ml_analysis_comprehensive.png"]

def version_of(pkg):
    try:
        return im.version(pkg)
    except im.PackageNotFoundError:
        return None

provenance = {
    "written": datetime.datetime.now().replace(microsecond=0).isoformat(),
    "author": "Max Brzezicki",
    "python": platform.python_version(),
    "packages": {p: version_of(p) for p in ("numpy", "pandas", "scipy", "statsmodels", "scikit-learn",
                                            "xgboost", "merf", "matplotlib", "seaborn", "tqdm", "fpdf2")},
    "inputs": {config.data_path: sha256(config.data_path)},
    "figures": {p: sha256(p) for p in FIGURE_FILES if os.path.exists(p)},
    "seed": RANDOM_SEED,
}
out = dict(MANIFEST)
out["_provenance"] = provenance
with open("results_manifest.json", "w") as f:
    json.dump(out, f, indent=2, default=str)
print(f"results_manifest.json written with {len(MANIFEST)} keys")
